# Xiaoyang Slides — 4 张图（一键导出）

只产 **slide 用的 4 张图**，1 对 1 对应你两页 slide：

| 文件 | 用在 |
|---|---|
| **fig1_imu_per_class.png**         | 页 1 上图（IMU Expert）|
| **fig2_imu_confusion.png**         | 页 1 下图（IMU Expert）|
| **fig3_phase_features_multi.png**  | 页 2 上图（Phase Arbitrator）⭐ |
| **fig4_alpha_comparison.png**      | 页 2 下图（Phase Arbitrator）⭐ untrained vs trained 对比 |

## 1. 挂 Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. 路径

In [ ]:
import os
MMAI = "/content/drive/MyDrive/MMAI"
INERTIAL = f"{MMAI}/utd_mhad/Inertial"

def find_first(name, dirs):
    for d in dirs:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    return None

IMU_CLF = find_first("imu_classifier_best.pt", [f"{MMAI}/pgmoe_ckpt", f"{MMAI}/Models"])
PGMOE_CKPT = find_first("pgmoe_best.pt", [f"{MMAI}/pgmoe_ckpt", f"{MMAI}/Models"])

REPO_FIG = "/content/drive/MyDrive/Multi-Modal-AI/project/final/figures/xiaoyang_slides"
SAVE_DIR = REPO_FIG if os.path.exists(os.path.dirname(REPO_FIG)) else f"{MMAI}/figures/xiaoyang_slides"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"IMU_CLF    : {IMU_CLF}")
print(f"PGMOE_CKPT : {PGMOE_CKPT or 'NOT FOUND (fig4 will fallback)'}")
print(f"SAVE_DIR   : {SAVE_DIR}")
assert IMU_CLF and os.path.exists(INERTIAL)

## 3. Imports + helper

In [ ]:
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMU_LEN = 192
UTD_LABELS = [
    "swipe left", "swipe right", "wave", "clap", "throw",
    "arm cross", "basketball shoot", "draw x",
    "draw circle CW", "draw circle CCW", "draw triangle",
    "bowling", "boxing", "baseball swing", "tennis swing",
    "arm curl", "tennis serve", "two hand push", "knock",
    "catch", "pickup and throw", "jogging", "walking",
    "sit to stand", "stand to sit", "forward lunge", "squat",
]

def load_imu(action, subject=1, trial=1):
    fp = f"{INERTIAL}/a{action}_s{subject}_t{trial}_inertial.mat"
    if not os.path.exists(fp): return None
    d = sio.loadmat(fp)["d_iner"].astype(np.float32)
    if d.shape[0] < IMU_LEN:
        d = np.concatenate([d, np.zeros((IMU_LEN - d.shape[0], 6), np.float32)], axis=0)
    return torch.from_numpy(d[:IMU_LEN]).T.contiguous()

## 4. 模型类

In [ ]:
class ResidualBlock1D(nn.Module):
    def __init__(self, in_c, out_c, kernel=5, stride=1):
        super().__init__()
        pad = kernel // 2
        self.conv = nn.Sequential(
            nn.Conv1d(in_c, out_c, kernel, stride=stride, padding=pad),
            nn.BatchNorm1d(out_c), nn.ReLU(inplace=True),
            nn.Conv1d(out_c, out_c, kernel, stride=1, padding=pad),
            nn.BatchNorm1d(out_c),
        )
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(nn.Conv1d(in_c, out_c, 1, stride=stride), nn.BatchNorm1d(out_c))
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.conv(x) + self.shortcut(x))

class IMUExpert(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(6, 64, 7, stride=2, padding=3), nn.BatchNorm1d(64), nn.ReLU(inplace=True))
        self.block1 = ResidualBlock1D(64, 128, 5, 2)
        self.block2 = ResidualBlock1D(128, 256, 5, 2)
        self.block3 = ResidualBlock1D(256, d_model, 3, 2)
    def forward(self, x):
        x = self.stem(x); x = self.block1(x); x = self.block2(x); x = self.block3(x)
        return x.transpose(1, 2)

class IMUClassifier(nn.Module):
    def __init__(self, num_classes=27, d_model=256):
        super().__init__()
        self.encoder = IMUExpert(d_model=d_model)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)
    def forward(self, x):
        return self.head(self.norm(self.encoder(x).mean(dim=1)))

class PhaseFeatures(nn.Module):
    def forward(self, x):
        acc = x[:, :3]
        mag = torch.sqrt((acc**2).sum(dim=1, keepdim=True) + 1e-8)
        mag_d = torch.diff(mag, dim=2, prepend=mag[:, :, :1])
        mag_dd = torch.diff(mag_d, dim=2, prepend=mag_d[:, :, :1])
        energy = (acc**2).sum(dim=1, keepdim=True)
        e_rate = torch.diff(energy, dim=2, prepend=energy[:, :, :1])
        return torch.cat([mag, mag_dd, e_rate], dim=1)

class PhaseArbitrator(nn.Module):
    def __init__(self, T_i=12):
        super().__init__()
        self.features = PhaseFeatures()
        self.pool = nn.AdaptiveAvgPool1d(T_i)
        self.encoder = nn.Sequential(
            nn.Conv1d(3, 64, 1), nn.ReLU(inplace=True),
            nn.Conv1d(64, 32, 1), nn.ReLU(inplace=True),
        )
        self.arbitrator = nn.Sequential(
            nn.Conv1d(32, 16, 1), nn.ReLU(inplace=True),
            nn.Conv1d(16, 1, 1), nn.Sigmoid(),
        )
    def forward(self, imu):
        return self.arbitrator(self.encoder(self.pool(self.features(imu)))).squeeze(1)

## 5. 跑 IMU classifier 在 test set 上（fig1 + fig2 共用）

In [ ]:
class IMUDataset(Dataset):
    def __init__(self, train=False):
        allowed = {1, 3, 5, 7} if train else {2, 4, 6, 8}
        self.samples = []
        for fn in sorted(os.listdir(INERTIAL)):
            if not fn.endswith("_inertial.mat"): continue
            p = fn.split("_")
            a, s = int(p[0][1:]), int(p[1][1:])
            if s not in allowed: continue
            self.samples.append((f"{INERTIAL}/{fn}", a - 1))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        fp, lb = self.samples[idx]
        d = sio.loadmat(fp)["d_iner"].astype(np.float32)
        if d.shape[0] < IMU_LEN:
            d = np.concatenate([d, np.zeros((IMU_LEN - d.shape[0], 6), np.float32)], axis=0)
        return torch.from_numpy(d[:IMU_LEN]).T.contiguous(), lb

imu_clf = IMUClassifier().to(device)
imu_clf.load_state_dict(torch.load(IMU_CLF, map_location=device, weights_only=False))
imu_clf.eval()

loader = DataLoader(IMUDataset(train=False), batch_size=32, shuffle=False)
preds, labels = [], []
with torch.no_grad():
    for x, y in loader:
        preds.extend(imu_clf(x.to(device)).argmax(1).cpu().numpy())
        labels.extend(y.numpy())
preds, labels = np.array(preds), np.array(labels)
imu_acc = accuracy_score(labels, preds)
imu_per_class = {c: (preds[labels == c] == c).mean() for c in range(27) if (labels == c).sum() > 0}
print(f"IMU test acc: {imu_acc*100:.2f}%")

## 6. Fig 1 — IMU per-class

In [ ]:
order = sorted(imu_per_class.items(), key=lambda r: r[1])
colors_bar = ['#EF4444' if a < 0.5 else '#F59E0B' if a < 0.8 else '#10B981' for _, a in order]

fig, ax = plt.subplots(figsize=(11, 9))
y_pos = np.arange(len(order))
ax.barh(y_pos, [a*100 for _, a in order], color=colors_bar, edgecolor='black', alpha=0.88)
for i, (c, a) in enumerate(order):
    ax.text(a*100 + 1, i, f'{a*100:.1f}%', va='center', fontsize=9)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"c{c}: {UTD_LABELS[c]}" for c, _ in order], fontsize=10)
ax.set_xlabel("Accuracy (%)", fontsize=12)
ax.set_xlim(0, 105)
ax.axvline(50, ls=":", color="red", alpha=0.4)
ax.axvline(80, ls=":", color="gray", alpha=0.4)
ax.set_title(f"IMU Expert: Per-Class Accuracy ({imu_acc*100:.2f}% overall)",
             fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
out1 = f"{SAVE_DIR}/fig1_imu_per_class.png"
plt.savefig(out1, dpi=200, bbox_inches='tight'); plt.show()
print(f"✅ {out1}")

## 7. Fig 2 — IMU confusion matrix

In [ ]:
cm = confusion_matrix(labels, preds, labels=list(range(27)))
fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax, shrink=0.85)

for i in range(27):
    for j in range(27):
        if cm[i, j] > 0:
            c = 'white' if cm[i, j] > cm.max() * 0.55 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=c, fontsize=8)

short = [l[:14] for l in UTD_LABELS]
ax.set_xticks(range(27)); ax.set_yticks(range(27))
ax.set_xticklabels(short, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(short, fontsize=8)
ax.set_xlabel("Predicted class", fontsize=12)
ax.set_ylabel("True class", fontsize=12)
ax.set_title(f"IMU Expert Confusion Matrix  (test acc {imu_acc*100:.2f}%)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
out2 = f"{SAVE_DIR}/fig2_imu_confusion.png"
plt.savefig(out2, dpi=200, bbox_inches='tight'); plt.show()
print(f"✅ {out2}")

## 8. Fig 3 — 多动作 phase features ⭐

In [ ]:
demo_actions = [
    (5,  "Throw (impact)",       "#EF4444"),
    (23, "Walking (periodic)",   "#3B82F6"),
    (1,  "Swipe L (smooth)",     "#10B981"),
    (10, "Draw circle CCW",      "#8B5CF6"),
]

pf = PhaseFeatures()
fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True)
feature_names = [r"$|a(t)|$", r"$d^2|a|/dt^2$", "energy rate"]

for col, (act, name, color) in enumerate(demo_actions):
    imu = load_imu(act).unsqueeze(0)
    feats = pf(imu)
    t = np.arange(IMU_LEN)
    for r in range(3):
        axes[r, col].plot(t, feats[0, r].numpy(), color=color, linewidth=1.6)
        axes[r, col].grid(True, alpha=0.2)
        axes[r, col].spines['top'].set_visible(False)
        axes[r, col].spines['right'].set_visible(False)
        if col == 0:
            axes[r, col].set_ylabel(feature_names[r], fontsize=12)
    axes[0, col].set_title(name, fontsize=13, fontweight='bold')
    axes[2, col].set_xlabel("timestep", fontsize=11)

fig.suptitle("Phase Features Differ Sharply Across Action Types\n(input to Phase Arbitrator)",
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
out3 = f"{SAVE_DIR}/fig3_phase_features_multi.png"
plt.savefig(out3, dpi=200, bbox_inches='tight'); plt.show()
print(f"✅ {out3}")

## 9. Fig 4 — α(t) untrained vs trained ⭐

In [ ]:
torch.manual_seed(42)
pa_un = PhaseArbitrator(T_i=12)

untrained = {}
for act, name, color in demo_actions:
    imu = load_imu(act).unsqueeze(0)
    with torch.no_grad():
        untrained[name] = (pa_un(imu).squeeze().numpy(), color)

trained = {}
if PGMOE_CKPT:
    try:
        sd = torch.load(PGMOE_CKPT, map_location='cpu', weights_only=False)
        if isinstance(sd, dict) and 'model_state' in sd:
            sd = sd['model_state']
        pa_keys = {k.replace('phase_arbitrator.', ''): v
                   for k, v in sd.items() if k.startswith('phase_arbitrator.')}
        if pa_keys:
            pa_tr = PhaseArbitrator(T_i=12)
            pa_tr.load_state_dict(pa_keys)
            pa_tr.eval()
            for act, name, color in demo_actions:
                imu = load_imu(act).unsqueeze(0)
                with torch.no_grad():
                    trained[name] = (pa_tr(imu).squeeze().numpy(), color)
            print(f"✅ Loaded trained phase_arbitrator from {PGMOE_CKPT}")
        else:
            print(f"⚠ {PGMOE_CKPT} 里没有 phase_arbitrator 权重")
    except Exception as e:
        print(f"⚠ Load failed: {e}")

if trained:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    for name, (a, c) in untrained.items():
        axes[0].plot(np.linspace(0, 1, 12), a, "o-", color=c, label=name, linewidth=2, markersize=7)
    for name, (a, c) in trained.items():
        axes[1].plot(np.linspace(0, 1, 12), a, "o-", color=c, label=name, linewidth=2, markersize=7)
    for ax in axes:
        ax.axhline(0.5, ls="--", color="gray", alpha=0.5)
        ax.set_ylim(0, 1)
        ax.set_xlabel("normalized time", fontsize=12)
        ax.grid(True, alpha=0.3); ax.legend(loc='best', fontsize=10)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    axes[0].set_title("Untrained (random init)", fontsize=13, fontweight='bold')
    axes[1].set_title("After PG-MoE joint training", fontsize=13, fontweight='bold')
    axes[0].set_ylabel(r"$\alpha$ (1=trust vision, 0=trust IMU)", fontsize=12)
    fig.suptitle(r"$\alpha(t)$: Phase Awareness Is Learned, Not Hardcoded",
                 fontsize=14, fontweight='bold', y=1.02)
else:
    print(f"⚠ fallback 只画 untrained")
    fig, ax = plt.subplots(figsize=(10, 5))
    for name, (a, c) in untrained.items():
        ax.plot(np.linspace(0, 1, 12), a, "o-", color=c, label=name, linewidth=2, markersize=8)
    ax.axhline(0.5, ls="--", color="gray", alpha=0.5)
    ax.set_ylim(0, 1)
    ax.set_xlabel("normalized time", fontsize=12)
    ax.set_ylabel(r"$\alpha$", fontsize=12)
    ax.set_title(r"$\alpha(t)$ — Untrained Baseline", fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3); ax.legend(loc='best')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
out4 = f"{SAVE_DIR}/fig4_alpha_comparison.png"
plt.savefig(out4, dpi=200, bbox_inches='tight'); plt.show()
print(f"✅ {out4}")

## 10. 打包下载

In [ ]:
import shutil
zip_path = "/content/xiaoyang_4figs.zip"
shutil.make_archive(zip_path.replace(".zip", ""), "zip", SAVE_DIR)
print("Bundle:")
for fn in sorted(os.listdir(SAVE_DIR)):
    sz = os.path.getsize(os.path.join(SAVE_DIR, fn)) / 1024
    print(f"  {fn}  ({sz:.1f} KB)")
print(f"\nzip: {os.path.getsize(zip_path)/1024:.1f} KB")

from google.colab import files
files.download(zip_path)

## 完事

zip 解压后 4 张图按顺序塞 slide：
- **fig1** + **fig2** → 页 1
- **fig3** + **fig4** → 页 2